In [ ]:
import torch

print("cuda:", torch.cuda.is_available(), "| torch", torch.__version__)

In [ ]:
# セットアップ：Drive のマウント、リポジトリの取得、runs/ を Drive に向ける（再実行しても壊れない）
import os
import subprocess
import sys

from google.colab import drive

REPO_URL = "https://github.com/karenmathtoml-oss/grokking-from-scratch.git"   # ← 自分のものに書き換える
REPO_DIR = "/content/grokking-from-scratch"
RUNS_DIR = "/content/drive/MyDrive/grokking-runs"


def sh(*cmd):
    """コマンドを実行して、出力をノートブックに表示する。失敗したら例外。"""
    result = subprocess.run(cmd, capture_output=True, text=True)
    print((result.stdout + result.stderr).strip())
    result.check_returncode()


drive.mount("/content/drive")

if os.path.isdir(REPO_DIR):
    sh("git", "-C", REPO_DIR, "pull")
else:
    sh("git", "clone", REPO_URL, REPO_DIR)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

os.makedirs(RUNS_DIR, exist_ok=True)
if not os.path.lexists("runs"):
    os.symlink(RUNS_DIR, "runs")        # runs/ の中身は Drive に残る

sh("pip", "install", "-q", "-e", ".[dev]")   # pyproject.toml の依存の下限を満たす
sh("git", "log", "-1", "--oneline")     # この実行が、どのコミットのコードかを記録する

In [ ]:
## バージョンの確認
!python -m pytest tests -q

## スモーク実行（所要時間の測定）

本番と同じ設定で `steps` だけを 2000 にします。`Wall time` から、本番（40000 ステップ）の所要時間を見積もります。
手元の CPU では 2000 ステップで約 242 秒でした。

In [ ]:
%%time
!python -m grokking.train --config configs/experiments/smoke.json

In [ ]:
# 出力先が空でないと再実行できないので、測り終えたら消す
!rm -rf runs/smoke

## 本番の学習

- 学習中は **タブを開いたまま** にする（セッションが切れると、最初からやり直し）
- 再実行するときは、先に `!rm -rf runs/base` で出力先を消す
- 結果は Drive の `grokking-runs/base/` に残る

In [ ]:
!python -m grokking.train --config configs/base.json

## 結果の確認

In [ ]:
import pandas as pd

df = pd.read_csv("runs/base/metrics.csv")
print(df.tail(3))
print()
print("テスト精度が初めて 0.5 を超えた行:")
print(df[df.test_accuracy > 0.5].head(1))